# Lesson 11 Lab — GPTQ: Second-Order Intuition and Layer Reconstruction

**Puzzle:** Why should two weights with the same magnitude receive different quantization treatment?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Nearest-weight quantization assumes every weight error matters equally. A linear layer disproves that assumption: inputs can excite some columns strongly and barely touch others, so the same weight perturbation can create very different output error. GPTQ uses approximate second-order information to organize that sensitivity during one-shot quantization.


## 0. Predict before running

1. Predict whether raw weight RMSE or held-out layer-output RMSE better matches the GPTQ objective.
2. Explain how input covariance makes two equal-magnitude weights differ in importance.
3. State why the notebook's sensitivity fallback is an intuition model rather than a GPTQ implementation.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

GPTQ reconstructs one layer at a time using the layer weights and representative input activations. It targets output distortion, not unweighted distance between original and rounded weights.

- Layer reconstruction minimizes output error under representative inputs, not raw weight error alone.
- Input covariance approximates which directions are sensitive.
- Production GPTQ uses structured second-order updates; a toy sensitivity model is not the library implementation.


## 2. Derive the mechanism

For weight error `ΔW` and inputs `X`, layer error is approximately `||XΔWᵀ||²`; the input Gram/Hessian approximation `XᵀX` weights sensitive directions. GPTQ uses inverse-Hessian information to compensate remaining weights as columns are quantized.

For a layer `Y=XWᵀ`, a weight perturbation ΔW produces `ΔY=XΔWᵀ`. The squared reconstruction loss is proportional to `||XΔWᵀ||²`, which can be written using the input Gram matrix `XᵀX`. This matrix is the local curvature signal: errors along frequently excited directions cost more than errors along quiet directions. GPTQ quantizes while using an approximate inverse Hessian to compensate remaining weights.

The notebook does not reproduce that sequential update. It uses an input-weighted error score to identify sensitive columns and preserves a fixed fraction in higher precision. That smaller construction isolates the central idea—optimize layer behavior, not the visual closeness of W—without claiming production GPTQ equivalence.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "11-gptq"
device = require_cuda()
torch.manual_seed(2026 + 11)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | naive group-wise INT4 applied uniformly to the layer weights |
| Candidate | INT4 plus a 12.5% input-sensitive column fallback |
| Held constant | same layer, calibration inputs, held-out inputs, quantizer, and fallback budget |
| Measurements | held-out output RMSE, MAE, cosine, max error, preserved fraction |
| Evidence | `numerical-model` |

**Experiment:** Compare naive INT4 weight quantization with a GPTQ-inspired sensitivity fallback that preserves columns with large input-weighted error.


## 5. Read the experiment code

The lab is deliberately GPTQ-inspired: it uses input-weighted sensitivity and a fallback to expose the objective, while clearly not claiming GPTQModel execution.

The experiment forms representative inputs, computes a naive quantized layer, estimates which columns create the largest input-weighted reconstruction cost, and restores only the highest-ranked columns. Both candidates are then evaluated on held-out inputs rather than on the calibration tensor used for ranking.

This makes the causal variable the allocation of a fixed precision budget. It still omits blockwise Hessian inversion, error propagation, act-order variants, packing, and a GPTQ runtime kernel, all of which are required for an end-to-end backend claim.

Only after these variables match the protocol should the cell be executed.


In [2]:
n,in_f,out_f=1024,256,192; x=torch.randn(n,in_f,device=device); x[:,::31]*=5; w=torch.randn(out_f,in_f,device=device)
ref=x@w.t(); _,_,naive=symmetric_quantize(w,bits=4,group_size=64); naive_out=x@naive.t()
sensitivity=x.square().mean(0)*((w-naive).square().mean(0)); keep=torch.topk(sensitivity,k=in_f//8).indices
aware=naive.clone(); aware[:,keep]=w[:,keep]; aware_out=x@aware.t()
result=base_result(11,"numerical-model"); result.update({"shape":[n,in_f,out_f],"preserved_column_fraction":round(len(keep)/in_f,4),
    "naive_output_error":error_metrics(ref,naive_out),"sensitivity_fallback_error":error_metrics(ref,aware_out),
    "conclusion":"Input-weighted sensitivity changed which quantization errors mattered; this is a GPTQ intuition model, not GPTQModel execution."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Naive INT4 RMSE | 2.324805 |
| Sensitivity fallback RMSE | 1.597145 |
| Naive cosine | 0.994488 |
| Fallback cosine | 0.997389 |
| Preserved columns | 12.5000% |


## 7. Interpret rather than merely print

Naive INT4 produced output RMSE 2.324805 and cosine 0.994488. Preserving 12.5% of columns selected by input-weighted sensitivity reduced RMSE to 1.597145 and raised cosine to 0.997389; MAE fell from 1.842271 to 1.269206.

The reduction demonstrates that equal storage bits can be allocated more intelligently when activation evidence is available. It does not show that this heuristic matches GPTQ quality, quantization time, or inference speed. Its value is to make the second-order objective observable in a small lab.

**Inspection rule:** Measure layer-output error on held-out inputs and label the experiment as an intuition model, not a GPTQ kernel benchmark.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA numerical experiment isolates an algorithmic mechanism. It is not the paper's complete implementation and does not establish a production kernel speedup.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Input-weighted sensitivity changed which quantization errors mattered; this is a GPTQ intuition model, not GPTQModel execution.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "numerical-model",
  "executed_at_utc": "2026-08-07T14:45:34+00:00",
  "lesson": 11,
  "naive_output_error": {
    "cosine": 0.99448818,
    "mae": 1.84227121,
    "max_abs": 12.09428406,
    "rmse": 2.32480502
  },
  "preserved_column_fraction": 0.125,
  "schema_version": 1,
  "sensitivity_fallback_error": {
    "cosine": 0.99738872,
    "mae": 1.26920569,
    "max_abs": 8.99761105,
    "rmse": 1.59714472
  },
  "shape": [
    1024,
    256,
    192
  ]
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> Second-order information changes the objective from nearest weights to faithful layer outputs.

**Acceptance/rollback:** Record calibration activations, damping, block/group size, ordering, layer reconstruction loss, end-task regression, and the deployed operator.

**Failure analysis:** Ranking on the final test inputs leaks evaluation and exaggerates robustness. Preserving columns also changes average bit width, so a fair comparison must report the precision budget. A low layer RMSE can still fail after nonlinearities or across a full model, and a good checkpoint can still be slow without a compatible packed kernel.


## 10. Extend the evidence

Replace the heuristic with a small sequential Hessian-aware quantizer and compare quantization order, damping, and block size. Then evaluate error layer by layer and after stacking several layers. Finally load a GPTQModel-compatible checkpoint in a serving backend and keep quantization quality separate from operator throughput.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
